# Engine: The Mass Gap — spectral residue of BAO

**Module:** `ValaQuenta/modules/bao_mass_gap`
**Wiki:** [wiki/bao_mass_gap.md](../../wiki/bao_mass_gap.md)

```
Δ = Ω_ζΣ − D* × ln(10) = 0.0007073575 = 1/(1000√2)
```

Two constants computed from opposite ends of the H_hat_RB operator. Their
difference is the gap. One value, zero free parameters.

This notebook walks the derivation step by step. Each step runs the engine and
prints the ordered operations the engine itself recorded, then the values those
operations produced.

**Step order**

| Step | What it establishes |
|------|--------------------|
| 1 | the two constants, from the canonical source |
| 2 | the subtraction — Δ > 0 |
| 3 | why Δ is a *residue*: the explicit formula at BAO scale |
| 4 | the closed form Δ = 1/(1000√2) |
| 5 | Δ against the measured acoustic scale |
| 6 | Δ as the compactification scale, 11 = 4 + 7 |
| 7 | all 7 checks |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

from ValaQuenta.modules.bao_mass_gap import maths as bmg
from ValaQuenta.modules.bao_mass_gap import BaoMassGapModule

engine = BaoMassGapModule()
print(f'{engine.display_name}  v{engine.version}')
print(f'{len(engine.formulary())} equations')

In [ ]:
def steps(result):
    # Print the ordered operations the engine recorded.
    for i, s in enumerate(result.get('derivation', []), start=1):
        print(f'{i:3d}. {s}')

def values(result, keys=None):
    # Print the scalar values the operations produced.
    items = [(k, v) for k, v in result.items()
             if not isinstance(v, (list, dict))
             and k not in ('derivation', 'latex')
             and (keys is None or k in keys)]
    w = max((len(k) for k, _ in items), default=0)
    for k, v in items:
        print(f'  {k:{w}s} = {v:.12f}' if isinstance(v, float) else f'  {k:{w}s} = {v}')

---
## Step 1 — The two constants

Both come from `ValaQuenta/engine/constants.py`, the canonical source. The
module imports them rather than redefining them.

```
Ω_ζΣ = 0.5671432904097838   Lambert W(1)  — thermal information ceiling
D*   = 0.24600              spectral ground state — BAO acoustic floor
```

Ω_ζΣ is the fixed point of the Lambert W function: `W(Ω) = Ω`, equivalently
`Ω · e^Ω = 1`. Confirm that first — if this identity does not hold to machine
precision, nothing downstream is worth reading.

In [ ]:
print(f'OMEGA_ZS = {bmg.OMEGA_ZS!r}')
print(f'D_STAR   = {bmg.D_STAR!r}')
print(f'LN10     = {bmg.LN10!r}')
print()

# The Lambert W fixed point:  Omega * e^Omega = 1
lhs = bmg.OMEGA_ZS * math.exp(bmg.OMEGA_ZS)
print(f'OMEGA_ZS * exp(OMEGA_ZS) = {lhs!r}')
print(f'deviation from 1         = {abs(lhs - 1.0):.3e}')
assert abs(lhs - 1.0) < 1e-15, 'Lambert W fixed point failed'
print('\nLambert W fixed point holds to machine precision.')

---
## Step 2 — The subtraction

`D*` is a spectral coordinate; `ln(10)` converts it into information units so
it can be compared against the ceiling. Then subtract.

The gap exists because the residue is positive.

In [ ]:
r_gap = bmg.gap_value()
steps(r_gap)
print()
values(r_gap)

In [ ]:
# Recompute independently of the module, from the definition alone.
floor   = bmg.D_STAR * math.log(10.0)
ceiling = bmg.OMEGA_ZS
gap     = ceiling - floor

print(f'floor   = D* * ln(10) = {floor!r}')
print(f'ceiling = OMEGA_ZS    = {ceiling!r}')
print(f'gap     = ceiling - floor = {gap!r}')
print()
print(f'matches module GAP: {gap == bmg.GAP}')
assert gap == bmg.GAP

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))

ax.hlines(bmg.BAO_CEILING, 0, 1, lw=2.5, color='#b8860b')
ax.hlines(bmg.BAO_FLOOR,   0, 1, lw=2.5, color='#2e8b57')
ax.fill_between([0, 1], bmg.BAO_FLOOR, bmg.BAO_CEILING, alpha=0.30, color='#c04040')

ax.annotate(f'ceiling  Ω_ζΣ = {bmg.BAO_CEILING:.7f}', (1.02, bmg.BAO_CEILING),
            va='center', fontsize=9, color='#b8860b')
ax.annotate(f'floor  D*·ln10 = {bmg.BAO_FLOOR:.7f}', (1.02, bmg.BAO_FLOOR),
            va='center', fontsize=9, color='#2e8b57')
ax.annotate(f'Δ = {bmg.GAP:.10f}', (0.5, (bmg.BAO_FLOOR + bmg.BAO_CEILING) / 2),
            ha='center', va='center', fontsize=11, weight='bold')

ax.set_xlim(0, 1.9); ax.set_xticks([])
ax.set_ylim(bmg.BAO_FLOOR - 0.0004, bmg.BAO_CEILING + 0.0004)
ax.set_ylabel('H_hat_RB information units')
ax.set_title('The gap is the band between floor and ceiling')
plt.tight_layout(); plt.show()

---
## Step 3 — Why Δ is a *residue*

This is the step the name of the engine refers to.

The explicit formula decomposes the prime distribution into a ground state plus
one standing wave per non-trivial zero:

```
ψ(x) = x − Σ_ρ x^ρ/ρ − ln(2π) − ½ln(1 − x⁻²)
       ▲   ▲
       │   └── spectral oscillations: one standing wave per γ_n
       └────── de Sitter expansion term: the ground state
```

Read at the BAO scale this is the acoustic spectrum of the CMB:

- the `x` term is the de Sitter expansion — the **acoustic ground state**
- `Σ_ρ x^ρ/ρ` is the acoustic oscillation set — **one standing wave per zero**
- the **residue** is what no standing wave absorbs

The natural BAO coordinate is `x_BAO = exp(1/Ω_ζΣ)` — the scale at which the
information ceiling is exactly one nat. Each zero contributes amplitude
`1/|ρ| = 1/√(¼+γ²)`, strictly decreasing in γ, so the first zero
`γ₁ = 14.134725` sets the largest single excitation above the ground state.

The residue is a difference of two constants. It does not depend on how many
zeros are summed — Step 3b demonstrates that directly.

In [ ]:
r_spec = bmg.spectral_residue()
steps(r_spec)

In [ ]:
values(r_spec, keys=['psi_ground_state', 'psi_spectral_sum', 'psi_correction',
                     'psi_computed', 'psi_chebyshev_exact', 'n_zeros_summed',
                     'x_bao', 'first_mode_amplitude',
                     'bao_floor', 'bao_ceiling', 'residue',
                     'residue_equals_gap'])
print()
print('The first 8 standing waves at x_BAO:')
print(f"  {'gamma_n':>12s} {'amplitude 1/|rho|':>18s} {'phase (rad)':>13s} {'cos':>9s}")
for w in r_spec['standing_waves']:
    print(f"  {w['gamma_n']:12.6f} {w['amplitude']:18.10f} "
          f"{w['phase_rad']:13.6f} {w['cos_at_bao']:9.5f}")

### Step 3b — the residue does not move with the number of zeros

Sum 1 zero or all 20. The spectral sum converges; the residue is a difference of
two constants and does not depend on it. This is what
`residue_is_n_independent` asserts.

In [ ]:
ns   = list(range(1, 21))
sums = [bmg.spectral_residue(n)['psi_spectral_sum'] for n in ns]
res  = [bmg.spectral_residue(n)['residue']          for n in ns]

print(f"  {'n_zeros':>8s} {'spectral sum':>16s} {'residue':>16s}")
for n, s, r in list(zip(ns, sums, res))[:5] + [('...', '', '')] + list(zip(ns, sums, res))[-2:]:
    if n == '...':
        print(f"  {'...':>8s}")
        continue
    print(f'  {n:8d} {s:16.10f} {r:16.10f}')

print()
print(f'residue spread over n = 1..20: {max(res) - min(res):.3e}')
assert max(res) - min(res) == 0.0, 'residue moved with n'
print('Residue is exactly invariant. Confirmed.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

ax1.plot(ns, sums, 'o-', ms=4, color='#3b6ea5')
ax1.set_xlabel('number of zeros summed'); ax1.set_ylabel('Σ_ρ Re(x^ρ/ρ) at x=10')
ax1.set_title('The spectral sum converges')

ax2.plot(ns, res, 'o-', ms=4, color='#c04040')
ax2.axhline(bmg.GAP, ls='--', lw=1, color='k')
ax2.set_xlabel('number of zeros summed'); ax2.set_ylabel('residue')
ax2.set_ylim(bmg.GAP - 1e-5, bmg.GAP + 1e-5)
ax2.set_title(f'The residue does not move   (Δ = {bmg.GAP:.10f})')

plt.tight_layout(); plt.show()

In [ ]:
# The zeros on the critical line, amplitude-weighted by 1/|rho|.
vd  = engine.viewer_data('spectral_residue', {}, 'complex_plane')
pts = vd['points']

fig, ax = plt.subplots(figsize=(4.2, 5.4))
ax.axvline(0.5, ls='--', lw=1, color='k', alpha=0.6)
ax.scatter([p['re'] for p in pts], [p['im'] for p in pts],
           s=[p['weight'] * 9000 for p in pts], alpha=0.55, color='#3b6ea5')
for p in pts:
    ax.annotate(f"γ={p['im']:.3f}", (p['re'] + 0.04, p['im']), fontsize=7, va='center')

ax.set_xlim(0, 1.1); ax.set_xlabel('Re(ρ)'); ax.set_ylabel('Im(ρ) = γ_n')
ax.set_title('Standing waves on σ=½\nmarker area ∝ 1/|ρ|')
plt.tight_layout(); plt.show()

print(f"floor   = {vd['marker']['floor']:.10f}")
print(f"ceiling = {vd['marker']['ceiling']:.10f}")
print(f"residue = {vd['marker']['residue']:.10f}")

---
## Step 4 — The closed form

```
Δ = 1/(1000√2) = 1/√(2×10⁶)
```

`1/√2 = sin(45°) = cos(45°)` — the point of maximum Red/Blue symmetry, where
the forward current equals the backward current. The √2 is the first
Cayley-Dickson doubling.

**Write it `1/(1000√2)` or `1/√(2×10⁶)`.** Not `1/√2000` — that is 0.02236,
which is 31.6× too large. Getting this wrong is the single easiest way to
misread the engine, so the next cell checks all three spellings side by side.

In [ ]:
r_id = bmg.gap_identity()
steps(r_id)
print()
values(r_id)

In [ ]:
candidates = {
    '1/sqrt(2000)'    : 1 / math.sqrt(2000),
    '1/(1000*sqrt(2))': 1 / (1000 * math.sqrt(2)),
    '1/sqrt(2e6)'     : 1 / math.sqrt(2_000_000),
}
print(f"  {'expression':20s} {'value':>18s} {'ratio to Δ':>14s}")
for name, val in candidates.items():
    flag = '  <-- WRONG, 31.6x too large' if val / bmg.GAP > 2 else ''
    print(f'  {name:20s} {val:18.12f} {val / bmg.GAP:14.6f}{flag}')

In [ ]:
# D* is carried to 5 decimals. Where does the identity pin it?
d_lo, d_hi = 0.24595, 0.24605
d_range = np.linspace(d_lo, d_hi, 400)
gaps    = bmg.OMEGA_ZS - d_range * bmg.LN10

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(d_range, gaps, lw=1.8, color='#3b6ea5', label='Δ(D*)')
ax.axhline(bmg.GAP_IDENTITY, ls='--', lw=1.2, color='#c04040', label='1/(1000√2)')
ax.axvline(bmg.D_STAR, ls=':', lw=1.2, color='k', label=f'carried D* = {bmg.D_STAR}')
ax.plot([r_id['d_star_for_exact']], [bmg.GAP_IDENTITY], 'o', ms=7, color='#c04040')
ax.annotate(f"exact at D* = {r_id['d_star_for_exact']:.10f}",
            (r_id['d_star_for_exact'], bmg.GAP_IDENTITY),
            textcoords='offset points', xytext=(10, -16), fontsize=8)

ax.set_xlabel('D*'); ax.set_ylabel('Δ')
ax.set_title(f"The identity pins D* to within {r_id['d_star_delta']:.2e}")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## Step 5 — Against the measured acoustic scale

Planck 2018 gives the sound horizon at the drag epoch as
`r_s = 147.09 ± 0.26 Mpc` — a fractional precision of 0.177%.

Δ sits at 0.40 of that error bar: above the noise floor, so it is a resolvable
feature of the acoustic spectrum.

In [ ]:
r_bao = bmg.bao_consistency()
steps(r_bao)
print()
values(r_bao)

---
## Step 6 — Δ as the compactification scale

`11 = 4 observable + 7 compact`. The compact 7 carry G₂ holonomy, and
`G₂ = Aut(𝕆)` — the automorphism group of the octonions. The 7 directions are
the imaginary octonion units `e₁..e₇`: algebraic, never spatial.

The compactification scale is Δ. Δ is computed, not tuned, so it is not a
modulus. No moduli, no landscape: `10^500 → 1`.

The dimension arithmetic is exact — the module carries it as `Fraction`, not
float, so `11 − 4 == 7` is an exact comparison rather than a float one.

In [ ]:
r_mt = bmg.mtheory_compactification()
steps(r_mt)
print()
values(r_mt)
print()
print(f'dimension types: MTHEORY_DIMS={type(bmg.MTHEORY_DIMS).__name__}, '
      f'OCTONION_IMAG_UNITS={type(bmg.OCTONION_IMAG_UNITS).__name__}')
print(f'exact: 11 - 4 == 7  ->  '
      f'{bmg.MTHEORY_DIMS - bmg.OBSERVABLE_DIMS == bmg.OCTONION_IMAG_UNITS}')

---
## Step 7 — All checks

Every claim above, run and reported.

In [ ]:
v = bmg.validate()
steps(v)
print()
for name, ok in v['checks'].items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print()
print(f"{sum(v['checks'].values())}/{v['n_checks']} pass   "
      f"free parameters: {v['free_parameters']}")
assert v['all_pass'], 'validation failed'

In [ ]:
# The console rendering — same text the curses GUI shows.
print(engine.viewer_data('summary', {}, 'text')['text'])

---
## Result

```
Ω_ζΣ       = 0.5671432904097838   thermal information ceiling
D*·ln10    = 0.5664359329         BAO acoustic ground state
Δ          = 0.0007073575         absorbed by no standing wave
1/(1000√2) = 0.0007071068         the Red/Blue symmetry point
```

7/7 checks pass. Zero free parameters.

Δ is consumed across the codebase as the compactification scale and as the
spectral floor constant. This module is where it is computed.

**See also**
- [wiki/bao_mass_gap.md](../../wiki/bao_mass_gap.md) — engine page
- [07_berry_keating.ipynb](07_berry_keating.ipynb) — where D* comes from
- [01_constants.ipynb](01_constants.ipynb) — Ω_ζΣ and the canonical constants